<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [12]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

Found existing installation: pydantic 1.10.26
Uninstalling pydantic-1.10.26:
  Successfully uninstalled pydantic-1.10.26
  Using cached pydantic-1.10.26-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (155 kB)
Using cached pydantic-1.10.26-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (2.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 1.4.8 requires pydantic<3.0.0,>=2.7.4, but you have pydantic 1.10.26 which is incompatible.
wandb 0.28.0 requires pydantic<3,>=2.6, but you have pydantic 1.10.26 which is incompatible.
thinc 8.3.13 requires pydantic<3.0.0,>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
weasel 1.0.0 requires pydantic>=2.0.0, but you have pydantic 1.10.26 which is incompatible.
google-genai 2.10.0 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 1.10.26 which is incompatible.

In [3]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [37]:
import os
import pandas as pd
import requests
import io

data_path = '/content/labelled_newscatcher_dataset.csv'
url = 'https://huggingface.co/datasets/okite97/news-data/resolve/main/labelled_newscatcher_dataset.csv?download=true'

print('Downloading dataset...')
try:
    response = requests.get(url, timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
    if response.status_code == 200:
        with open(data_path, 'wb') as f:
            f.write(response.content)
        print(f'Download successful! File size: {os.path.getsize(data_path)} bytes.')
    else:
        print(f'Failed to download (Status {response.status_code}). Generating expanded dummy data...')
        # Expanded dummy data for better similarity search demo
        dummy_data = "topic;link;domain;published_date;title;lang;topic_area;id\n" \
                     "SCIENCE;http://news.com;news.com;2021-01-01;New discovery in deep space exploration;en;Science;0\n" \
                     "TECHNOLOGY;http://tech.com;tech.com;2021-01-01;New AI model released by major lab;en;Tech;1\n" \
                     "BUSINESS;http://biz.com;biz.com;2021-01-01;Stock market hits record high this morning;en;Business;2\n" \
                     "SCIENCE;http://sci.com;sci.com;2021-01-02;Mars rover finds signs of ancient water;en;Science;3\n" \
                     "TECHNOLOGY;http://gadget.com;gadget.com;2021-01-02;New smartphone battery lasts a week;en;Tech;4\n" \
                     "WORLD;http://world.com;world.com;2021-01-02;Global summits focus on climate change;en;World;5"
        with open(data_path, 'w') as f:
            f.write(dummy_data)

    pdf = pd.read_csv(data_path, sep=';')
    if 'id' not in pdf.columns:
        pdf['id'] = range(len(pdf))

    print(f'Dataset loaded successfully with {len(pdf)} rows.')
    display(pdf.head())
    pdf_subset = pdf.head(1000)
    print('Created pdf_subset.')
except Exception as e:
    print(f'Critical error: {e}')

Failed to download (Status 404). Generating expanded dummy data...
Dataset loaded successfully with 6 rows.


,topic,link,domain,published_date,title,lang,topic_area,id
0,SCIENCE,http://news.com,news.com,2021-01-01,New discovery in deep space exploration,en,Science,0
1,TECHNOLOGY,http://tech.com,tech.com,2021-01-01,New AI model released by major lab,en,Tech,1
2,BUSINESS,http://biz.com,biz.com,2021-01-01,Stock market hits record high this morning,en,Business,2
3,SCIENCE,http://sci.com,sci.com,2021-01-02,Mars rover finds signs of ancient water,en,Science,3
4,TECHNOLOGY,http://gadget.com,gadget.com,2021-01-02,New smartphone battery lasts a week,en,Tech,4


Created pdf_subset.


## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [38]:
def example_create_fn(idx: int, text: str) -> InputExample:
    return InputExample(guid=str(idx), texts=[text], label=0.0)

# Create training examples from the updated subset data
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]
print(f'Created {len(faiss_train_examples)} training examples.')

Created 6 training examples.


In [39]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
print(f'Embedding shape: {faiss_title_embedding.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (6, 384)


## 🌟 Exercise 3 · FAISS indexing and search

In [40]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.copy().astype('float32')
faiss.normalize_L2(content_encoded_normalized)

index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f'FAISS index updated with {index_content.ntotal} documents.')

FAISS index updated with 6 documents.


In [25]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # Encode the query using the same sentence transformer model
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vector)

    # Search the index
    sims, ids = index_content.search(query_vector, k)

    # Retrieve matching rows and add similarity scores
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()
    results['similarities'] = sims[0][:len(results)] # Handle cases where fewer than k matches exist
    return results.sort_values(by='similarities', ascending=False)

print('Search results for "space":')
display(search_content('space', pdf_to_index, k=2))

Search results for "space":


,topic,link,domain,published_date,title,lang,topic_area,id,similarities
0,SCIENCE,http://news.com,news.com,2021-01-01,New discovery in space,en,Science,0,0.543548
1,TECHNOLOGY,http://tech.com,tech.com,2021-01-01,New AI model released,en,Tech,1,0.106491


## 🌟 Exercise 4 · ChromaDB collection and querying

In [41]:
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)

collection.add(
    documents=pdf_subset['title'].tolist(),
    metadatas=[{'topic': row['topic']} for _, row in pdf_subset.iterrows()],
    ids=[str(i) for i in pdf_subset['id']]
)

query_text = 'What is the latest in space exploration?'
results = collection.query(query_texts=[query_text], n_results=2)
print(f'Query: {query_text}')
print(json.dumps(results, indent=2))

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event client_start: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.posthog:Failed to send telemetry event collection_add: capture() takes 1 positional argument but 3 were given


Query: What is the latest in space exploration?
{
  "ids": [
    [
      "0",
      "1"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "New discovery in deep space exploration",
      "New AI model released by major lab"
    ]
  ],
  "metadatas": [
    [
      {
        "topic": "SCIENCE"
      },
      {
        "topic": "TECHNOLOGY"
      }
    ]
  ],
  "distances": [
    [
      0.7318510413169861,
      1.3829727172851562
    ]
  ]
}


## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [42]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_id = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_id)

question = "What is the latest news about space and Mars?"
# Use the documents retrieved from the updated ChromaDB collection
context_docs = results['documents'][0]
context = ' '.join(context_docs)

# Construct prompt
prompt = f"Answer the following question using the provided context.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"

# Tokenize input
inputs = tokenizer(prompt, return_tensors='pt')

# Generate output
output_tokens = model_t5.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

# Decode response
response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print(f'Question: {question}')
print(f'Retrieved Context: {context}')
print(f'AI Response: {response}')

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Question: What is the latest news about space and Mars?
Retrieved Context: New discovery in deep space exploration New AI model released by major lab
AI Response: New AI model
